<a href="https://colab.research.google.com/github/tal21-linares/curso-ia-para-economia/blob/main/Copia_de_6_Tercer_Parcial_Seleccion_Mejor_Modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/LinaMariaCastro/curso-ia-para-economia/blob/main/clases/5_Aprendizaje_supervisado/6_Competencia_Seleccion_Mejor_Modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Inteligencia Artificial con Aplicaciones en Economía I**

- 👩‍🏫 **Profesora:** [Lina María Castro](https://www.linkedin.com/in/lina-maria-castro)  
- 📧 **Email:** [lmcastroco@gmail.com](mailto:lmcastroco@gmail.com)  
- 🎓 **Universidad:** Universidad Externado de Colombia - Facultad de Economía

# 🏆 **Tercer parcial: Selección del Mejor Modelo**

**ESTÁ PROHIBIDO EL USO DE GRANDES MODELOS DE LENGUAJE COMO CHATGPT, CLAUDE, GEMINI, ENTRE OTROS, PARA RESOLVER ESTE EJERCICIO**

**Trabajo en grupos de 3**

**Objetivo:** Predecir las ventas de una compañía (`Sales`) teniendo en cuenta su inversión en publicidad.

**Dataset:** `train_df_ventas.csv` disponible en el repositorio del curso.

**IMPORTANTE: Los datos cargados solo corresponden a `train`.**

**Metodología:**
1.  Cargar y explorar los datos.
2.  Preprocesar los datos si es necesario.
3.  De los siguientes modelos, entrenar por lo menos 2:
    - Regresión Lineal
    - Regresión Polinómica
    - KNN Regressor
    - Decision Tree Regressor
    - Random Forest Regressor
    - Gradient Boosting Regressor
    - XGBoost Regressor
4.  Si lo considera necesario, usar `GridSearchCV` con Validación Cruzada (`cv=5`) para optimizar los hiperparámetros. La métrica de optimización debe ser el **RMSE** (Root Mean Squared Error), por lo que debe usar `scoring='neg_root_mean_squared_error'` (el valor será negativo y se multiplicará por -1 al final).
5.  Comparar los modelos y seleccionar el mejor, teniendo en cuenta el menor RMSE.

**Forma de entrega:**

- Nombrar el archivo de la siguiente forma: “Tercer_Parcial_apellidos.ipynb”.
- Suba el Jupyter Notebook a su cuenta en Github y envíe el link en el siguiente Forms: https://forms.cloud.microsoft/r/Bsy2U83tbc. No olvide indicar claramente cuál es el modelo seleccionado.

**IMPORTANTE:** No se recibirán talleres en Google Colab, el notebook debe estar subido en Github.

**Calificación**

La docente, evaluará el modelo seleccionado por ustedes en el `test set`.

El proceso seguirá estas reglas:

- **Criterio de Ganador:** El equipo que tenga todo el procedimiento correcto y obtenga el Root Mean Squared Error (RMSE) más bajo en el test set recibirá una calificación de 5.0.

- **Criterio de Desempate:** En caso de empate en el RMSE, se otorgará la ventaja al equipo que haya entrenado y evaluado más modelos.

- **Escalafón de Notas:** A partir del primer puesto, se restará 0.1 a la nota final por cada posición inferior (2º lugar: 4.9, 3er lugar: 4.8, etc.).

- **Validación de Procedimiento:** Es obligatorio que el código sea reproducible por la docente (no olivde colocar las semillas en los procesos aleatorios). Si el script contiene errores, el equipo quedará fuera de la competencia y se dará una calificación acorde a lo que esté correcto.

**Explicación de las variables:**

- Sales: Ventas (millones USD). --> **Esta es la variable objetivo**
- TV: Gasto en promoción televisiva (millones USD).
- Radio: Gasto en promoción radiofónica (millones USD).
- Social Media: Gasto en promoción en redes sociales (millones USD).
- Influencer: Indica si la promoción se realizó en colaboración con Mega, Macro, Nano o Micro influencers.


**Nombres estudiantes del equipo:**



*   Sebastian Casas
*   Alejandro Velandia
*   Talia Linares






# **Desarrollo**

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Métricas y Tuning
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

### Mejorar visualización de dataframes y gráficos

In [ ]:

# Que muestre todas las columnas
pd.options.display.max_columns = None
# En los dataframes, mostrar los float con dos decimales
pd.options.display.float_format = '{:,.2f}'.format

# Configuraciones para una mejor visualización
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

###Cargar los datos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = '/content/drive/MyDrive/Data/'

In [ ]:
# Para establecer el directorio de los archivos
os.chdir(path)

In [ ]:
df = pd.read_csv('train_df_ventas.csv')
df.head()

,TV,Social Media,Influencer,Radio,Sales
0,17.58,1.22,Macro,22.28,173.61
1,14.59,5.48,Micro,22.10,140.54
2,25.18,2.28,Mega,22.20,197.35
3,12.90,1.83,Nano,21.74,184.05
4,28.38,4.34,Nano,22.61,318.63


###Exploración de los datos

In [ ]:
print("Información del DataFrame:")
df.info()

Información del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3636 entries, 0 to 3635
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   TV            3636 non-null   float64
 1   Social Media  3636 non-null   float64
 2   Influencer    3636 non-null   object 
 3   Radio         3636 non-null   float64
 4   Sales         3636 non-null   float64
dtypes: float64(4), object(1)
memory usage: 142.2+ KB


Preprocesamiento

In [ ]:
X = df.drop('Sales', axis=1)
y = df['Sales']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Tamaño de X_train: {X_train.shape}")
print(f"Tamaño de X_test: {X_test.shape}")

Tamaño de X_train: (2908, 4)
Tamaño de X_test: (728, 4)


In [ ]:
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

print(f"Columnas Numéricas: {list(numerical_features)}")
print(f"Columnas Categóricas: {list(categorical_features)}")

Columnas Numéricas: ['TV', 'Social Media', 'Radio']
Columnas Categóricas: ['Influencer']


In [ ]:
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

In [ ]:
# Crear los transformadores
numeric_transformer = StandardScaler()


In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print(f"\nForma de X_train procesado: {X_train_processed.shape}")
print(f"Forma de X_test procesado: {X_test_processed.shape}")


Forma de X_train procesado: (2908, 7)
Forma de X_test procesado: (728, 7)


Random Forest

In [36]:
modelo_gb = GradientBoostingRegressor(random_state=42)

modelo_gb.fit(X_train_processed, y_train)

y_pred_gb = modelo_gb.predict(X_test_processed)

rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_gb))
r2_gb = r2_score(y_test, y_pred_gb)

print(f"RMSE Gradient Boosting: {rmse_gb:.2f}")
print(f"R2 Gradient Boosting: {r2_gb:.4f}")

RMSE Gradient Boosting: 42.47
R2 Gradient Boosting: 0.7885


In [37]:
modelo_rf = RandomForestRegressor(random_state=42)

modelo_rf.fit(X_train_processed, y_train)

y_pred_rf = modelo_rf.predict(X_test_processed)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print(f"RMSE Random Forest: {rmse_rf:.2f}")
print(f"R2 Random Forest: {r2_rf:.4f}")

RMSE Random Forest: 43.89
R2 Random Forest: 0.7741


In [44]:
resultados = pd.DataFrame([
    ["Random Forest", rmse_rf],
    ["Gradient Boosting", rmse_gb]
], columns=['Modelo', 'RMSE'])

resultados = resultados.sort_values(by='RMSE')

print(resultados)

              Modelo  RMSE
1  Gradient Boosting 42.47
0      Random Forest 43.89


# **Indica claramente cuál modelo seleccionaste como el mejor**

El modelo Gradient Boosting presentó un mejor desempeño predictivo frente a Random Forest, ya que obtuvo un menor RMSE (42.47) y un mayor coeficiente de determinación R² (0.7885). Esto indica que el modelo comete menos errores en sus predicciones y logra explicar una mayor proporción de la variabilidad de la variable objetivo, por lo que se considera el modelo más adecuado para este problema de regresión.